In [1]:
%pip install opencv-python imageio mediapipe kagglehub

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
# import torchvision

import cv2
# import tensorflow as tf

import numpy as np
from typing import List
from matplotlib import pyplot as plt
import imageio


In [3]:
from processing.video_loader import VideoLoader
from processing.preprocessor import Preprocessor

# building data loading functions

download dataset from kaggle
https://www.kaggle.com/datasets/mohamedbentalb/lipreading-dataset and save them in data\lips-dataset

In [4]:
%pip install -q kagglehub

import shutil
from pathlib import Path
import kagglehub

DATASET_HANDLE = 'mohamedbentalb/lipreading-dataset'
target_dir = Path('data/lips-dataset')

base = Path(kagglehub.dataset_download(DATASET_HANDLE))
for candidate in (base, base / 'data'):
    if (candidate / 's1').is_dir() or (candidate / 'alignments').is_dir():
        base = candidate
        break

print(f'Downloaded to: {base}')

target_dir.mkdir(parents=True, exist_ok=True)
for name in ('s1', 'alignments'):
    src = base / name
    dst = target_dir / name
    if not src.exists():
        print(f'SKIP: {name} not found in {base}')
    elif dst.exists():
        print(f'SKIP: {name} already at {dst}')
    else:
        shutil.move(str(src), str(dst))
        print(f'Moved {name} -> {dst}')


Note: you may need to restart the kernel to use updated packages.
Downloaded to: /home/lbyubuntu/.cache/kagglehub/datasets/mohamedbentalb/lipreading-dataset/versions/1
SKIP: s1 not found in /home/lbyubuntu/.cache/kagglehub/datasets/mohamedbentalb/lipreading-dataset/versions/1
SKIP: alignments not found in /home/lbyubuntu/.cache/kagglehub/datasets/mohamedbentalb/lipreading-dataset/versions/1


In [5]:
#test_path = r'.\data\lips-dataset\s1\bbaf2n.mpg' ## Windows File Directory
test_path = r'./data/lips-dataset/s1/bbaf2n.mpg' ## Macbook File Directory
video_loader = VideoLoader()
video_data = video_loader.load(test_path)
# video_data

I0000 00:00:1786867402.026107  183952 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1786867402.028900  184059 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.1.3), renderer: AMD Radeon RX 7800 XT (radeonsi, navi32, ACO, DRM 3.64, 7.0.0-29-generic)
W0000 00:00:1786867402.029281  183952 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1786867402.036831  184073 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786867402.056779  184065 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [6]:
preprocessor = Preprocessor()
video_data = preprocessor.convert_to_greyscale(video_data)

if fail to crop lips, head over to google https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task



In [7]:
# cv2.imshow("greyscale", video_data["frames"][0])
# cv2.waitKey(0)
# cv2.destroyAllWindows()

In [8]:
vocab = [x for x in "abcdefghijklmnopqrstuvwxyz'?!123456789 "]

In [9]:
char_to_num = {char: idx for idx, char in enumerate(vocab)}
num_to_char = {idx: char for char, idx in char_to_num.items()}

print(f"The vocabulary is: {vocab} (size = {len(vocab)})")

The vocabulary is: ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', "'", '?', '!', '1', '2', '3', '4', '5', '6', '7', '8', '9', ' '] (size = 39)


In [10]:
def encode_text(text):
    return torch.tensor([char_to_num[c] for c in text], dtype=torch.long)

def decode_indices(indices):
    return "".join(num_to_char[int(i)] for i in indices)

x = encode_text("hello")
print(x)
print(decode_indices(x))

tensor([ 7,  4, 11, 11, 14])
hello


In [11]:
def load_alignments(path:str) -> List[str]: 
    with open(path, 'r') as f: 
        lines = f.readlines() 
    tokens = []
    for line in lines:
        line = line.split()
        if line[2] != 'sil': 
            tokens = [*tokens,' ',line[2]]
    return [encode_text(x) for x in tokens]

def load_data(path: str) -> dict: 
    # path = bytes.decode(path.numpy())
    #file_name = path.split('/')[-1].split('.')[0]
    # File name splitting for windows
    file_name = os.path.basename(path).split('.')[0]
    video_path = os.path.join('data','lips-dataset','s1',f'{file_name}.mpg')
    alignment_path = os.path.join('data', 'lips-dataset', 'alignments','s1',f'{file_name}.align')
    video_data = video_loader.load(path) 
    alignments = load_alignments(alignment_path)
    video_data['alignments'] = alignments
    return video_data


video_data = load_data(test_path)
# print(video_data)
print(video_data['alignments'])
print([decode_indices(x) for x in video_data['alignments']])

I0000 00:00:1786867402.984493  183952 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1786867402.985361  184121 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.1.3), renderer: AMD Radeon RX 7800 XT (radeonsi, navi32, ACO, DRM 3.64, 7.0.0-29-generic)
W0000 00:00:1786867402.985620  183952 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
W0000 00:00:1786867402.989964  184133 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786867403.001176  184127 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


[tensor([38]), tensor([ 1,  8, 13]), tensor([38]), tensor([ 1, 11, 20,  4]), tensor([38]), tensor([ 0, 19]), tensor([38]), tensor([5]), tensor([38]), tensor([19, 22, 14]), tensor([38]), tensor([13, 14, 22])]
[' ', 'bin', ' ', 'blue', ' ', 'at', ' ', 'f', ' ', 'two', ' ', 'now']


# Creating data pipeline

In [12]:
samples  = []
word_dict = {}
for i in os.listdir('data/lips-dataset/s1'):
    if i.endswith('.mpg'):
        print(i)
        video_data = load_data(os.path.join('data/lips-dataset/s1', i))
        samples.append((video_data['frames'], video_data['alignments']))
        
        print(video_data['alignments'])
        sentence = [decode_indices(x) for x in video_data['alignments']]
        print(sentence)
        for word in sentence: 
            if word not in word_dict: 
                word_dict[word] = 1
            else: 
                word_dict[word] += 1
    break

print(sorted(word_dict.items(), key=lambda x: x[1], reverse=True))

swbo8n.mpg


I0000 00:00:1786867403.813795  183952 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1786867403.814998  184160 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.1.3), renderer: AMD Radeon RX 7800 XT (radeonsi, navi32, ACO, DRM 3.64, 7.0.0-29-generic)
W0000 00:00:1786867403.815406  183952 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
W0000 00:00:1786867403.818408  184174 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786867403.828754  184163 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


[tensor([38]), tensor([18,  4, 19]), tensor([38]), tensor([22,  7,  8, 19,  4]), tensor([38]), tensor([ 1, 24]), tensor([38]), tensor([14]), tensor([38]), tensor([ 4,  8,  6,  7, 19]), tensor([38]), tensor([13, 14, 22])]
[' ', 'set', ' ', 'white', ' ', 'by', ' ', 'o', ' ', 'eight', ' ', 'now']
[(' ', 6), ('set', 1), ('white', 1), ('by', 1), ('o', 1), ('eight', 1), ('now', 1)]


In [13]:
#cv2.imshow("test", video_data['frames'][40])
#cv2.waitKey(0)
#cv2.destroyAllWindows()

## Alignment Algorithm
### LBY: The cell below runs the align_videos.py as if it was in a terminal, input arguments include data directory and output directory
### It is important to note to save the kaggle dataset in its subfolder called data/lips-dataset, of which should have the following subfolders:
### (a) alignments, (b) s1, (c) processedData [This is an addition]

In [ ]:
!python align_videos.py --video-dir data/lips-dataset/s1 --align-dir data/lips-dataset/alignments/s1 --output-dir data/lips-dataset/processedData --grayscale --crop-mouth

## Train/Test Split & Model Evaluation
### LBY: Split `aligned_words.csv` (word-level, stratified 80/20), train a LipNet-style classifier, and report test accuracy.


In [15]:
from dataset import get_word_classification_dataloaders
from model import LipReadingClassifier

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_loader, test_loader, word_to_index, index_to_word = get_word_classification_dataloaders(
    test_size=0.2, seed=42, batch_size=32, stratify=True
)
num_classes = len(word_to_index)

print(f'Device: {device}')
print(f'Train samples: {len(train_loader.dataset)}')
print(f'Test samples: {len(test_loader.dataset)}')
print(f'Classes: {num_classes}')


Device: cuda
Train samples: 4796
Test samples: 1204
Classes: 51


In [17]:
from pathlib import Path

import torch
import torch.nn as nn

from train import set_seed, train_one_epoch, evaluate

set_seed(42)

model = LipReadingClassifier(num_classes=num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

out_dir = Path('checkpoints')
out_dir.mkdir(parents=True, exist_ok=True)

epochs = 100  
best_accuracy = 0.0
for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    print(
        f'Epoch {epoch:2d}/{epochs} | '
        f'train_loss {train_loss:.4f} train_acc {train_acc:.4f} | '
        f'test_loss {test_loss:.4f} test_acc {test_acc:.4f}'
    )

    if test_acc > best_accuracy:
        best_accuracy = test_acc
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'word_to_index': word_to_index,
            'test_accuracy': test_acc,
        }
        torch.save(checkpoint, out_dir / 'best.pt')

print(f'Best test accuracy: {best_accuracy:.4f}')


Epoch  1/100 | train_loss 3.2914 train_acc 0.1076 | test_loss 3.0299 test_acc 0.1370
Epoch  2/100 | train_loss 3.0332 train_acc 0.1312 | test_loss 2.8993 test_acc 0.1553
Epoch  3/100 | train_loss 2.9812 train_acc 0.1407 | test_loss 2.8879 test_acc 0.1686
Epoch  4/100 | train_loss 2.9395 train_acc 0.1572 | test_loss 2.8784 test_acc 0.1445
Epoch  5/100 | train_loss 2.9154 train_acc 0.1555 | test_loss 2.8580 test_acc 0.1603
Epoch  6/100 | train_loss 2.8869 train_acc 0.1714 | test_loss 2.8345 test_acc 0.1669
Epoch  7/100 | train_loss 2.8960 train_acc 0.1612 | test_loss 2.8465 test_acc 0.1603
Epoch  8/100 | train_loss 2.8796 train_acc 0.1691 | test_loss 2.8384 test_acc 0.1611
Epoch  9/100 | train_loss 2.8860 train_acc 0.1683 | test_loss 2.8358 test_acc 0.1628
Epoch 10/100 | train_loss 2.8833 train_acc 0.1693 | test_loss 2.8484 test_acc 0.1628
Epoch 11/100 | train_loss 2.8851 train_acc 0.1653 | test_loss 2.8284 test_acc 0.1678
Epoch 12/100 | train_loss 2.8689 train_acc 0.1681 | test_loss 2.8

In [18]:
# Optionally load the checkpoint saved above and re-evaluate.
checkpoint = torch.load('checkpoints/best.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
saved_epoch = checkpoint['epoch']
saved_acc = checkpoint['test_accuracy']
print(f'Loaded checkpoint (epoch {saved_epoch}, saved test acc {saved_acc:.4f})')
print(f'Re-evaluated test accuracy: {evaluate(model, test_loader, criterion, device)[1]:.4f}')


Loaded checkpoint (epoch 91, saved test acc 0.8522)
Re-evaluated test accuracy: 0.8522
